# CERA-Bench generation

In [ ]:

from __future__ import annotations
import os, sys, json, math, random, hashlib, shutil, subprocess, importlib.util, time
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

RUN_MODE = "debug"
TEST_MODE = os.environ.get("CERA_TEST_MODE", "0") == "1"
SEEDS = [42, 43, 44]
DEBUG_TASKS = 3
FULL_TASKS = 120
MODEL_SPECS = [
    {"label": "qwen3-8b", "model_id": "Qwen/Qwen3-8B"},
    {"label": "qwen3-14b", "model_id": "Qwen/Qwen3-14B"},
    {"label": "qwen3-32b", "model_id": "Qwen/Qwen3-32B"},
    {"label": "llama3.1-8b-instruct", "model_id": "meta-llama/Llama-3.1-8B-Instruct"},
]
DECISION_TEMPERATURE = 0.85
MIN_HORIZON = 2
MAX_ABS_LABEL_BIAS_MARGIN = 8.0
env_root = os.environ.get("CERA_ROOT")
candidates = [Path(env_root)] if env_root else []
candidates += [Path("/kaggle/working/cera"), Path("/mnt/data/cera"), Path.cwd()]
ROOT = next((p for p in candidates if p.exists() and (p.name == "cera" or (p / "configs").exists())), Path.cwd())
BENCHMARK_ROOT = ROOT / ("benchmark_debug" if RUN_MODE == "debug" else "benchmark")
TASKS_DIR = BENCHMARK_ROOT / "tasks"
TRACES_DIR = BENCHMARK_ROOT / "traces"
ORACLES_DIR = BENCHMARK_ROOT / "oracles"
MANIFESTS_DIR = BENCHMARK_ROOT / "manifests"
for p in [TASKS_DIR, TRACES_DIR, ORACLES_DIR, MANIFESTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
if RUN_MODE not in {"debug", "full"}:
    raise ValueError("RUN_MODE must be 'debug' or 'full'.")
TASK_COUNT = DEBUG_TASKS if RUN_MODE == "debug" else FULL_TASKS
ACTIVE_MODELS = MODEL_SPECS if RUN_MODE == "full" else MODEL_SPECS
ACTIVE_SEEDS = [42] if RUN_MODE == "debug" else SEEDS
print(f"mode={RUN_MODE} tasks={TASK_COUNT} models={len(ACTIVE_MODELS)} seeds={len(ACTIVE_SEEDS)} cases={TASK_COUNT*len(ACTIVE_MODELS)*len(ACTIVE_SEEDS)}")


In [ ]:
if not TEST_MODE:
    for module, package in [("transformers","transformers"),("bitsandbytes","bitsandbytes")]:
        if importlib.util.find_spec(module) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [ ]:

def stable_hash(value):
    return int(hashlib.sha256(str(value).encode()).hexdigest()[:12], 16)
def rng_for(*parts):
    return np.random.default_rng(stable_hash("::".join(map(str, parts))))
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

WORKLOADS = [
    ("patch_test_publish","software",[("TEST",1),("BUILD",1),("VALIDATE",1),("PUBLISH",1),("TEST",1),("BUILD",1),("VALIDATE",1),("PUBLISH",1)]),
    ("build_release","software",[("BUILD",1),("TEST",1),("VALIDATE",1),("PUBLISH",1),("BUILD",1),("TEST",1),("VALIDATE",1),("PUBLISH",1)]),
    ("dependency_update_build","software",[("BUILD",1),("VALIDATE",1),("TEST",1),("PUBLISH",1),("BUILD",1),("VALIDATE",1),("TEST",1),("PUBLISH",1)]),
    ("artifact_promotion","deployment",[("PUBLISH",1),("VALIDATE",1),("PROMOTE",1),("APPROVE",1),("PUBLISH",1),("VALIDATE",1),("PROMOTE",1),("APPROVE",1)]),
    ("config_rollout","deployment",[("PUBLISH",1),("VALIDATE",1),("PROMOTE",1),("ROLLOUT",1),("PUBLISH",1),("VALIDATE",1),("PROMOTE",1),("ROLLOUT",1)]),
    ("staged_release","deployment",[("PUBLISH",1),("VALIDATE",1),("PROMOTE",1),("APPROVE",1),("ROLLOUT",1),("PROMOTE",1),("APPROVE",1),("ROLLOUT",1)]),
    ("etl_publish","data",[("VALIDATE",1),("PUBLISH",1),("UPDATE",1),("COMMIT",1),("VALIDATE",1),("PUBLISH",1),("UPDATE",1),("COMMIT",1)]),
    ("validation_publish","data",[("UPDATE",1),("VALIDATE",1),("TEST",1),("PUBLISH",1),("UPDATE",1),("VALIDATE",1),("TEST",1),("PUBLISH",1)]),
    ("incremental_pipeline","data",[("READ",1),("UPDATE",1),("VALIDATE",1),("PUBLISH",1),("READ",1),("UPDATE",1),("VALIDATE",1),("PUBLISH",1)]),
    ("reservation_approval","transactional",[("APPEND",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1),("APPROVE",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1)]),
    ("versioned_record_update","transactional",[("APPEND",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1),("READ",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1)]),
    ("multi_step_state_transition","transactional",[("APPEND",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1),("APPROVE",1),("UPDATE",1),("VALIDATE",1),("COMMIT",1)]),
]
assert len(WORKLOADS) == 12


In [ ]:

def make_task_specs(n=120):
    specs=[]
    for workload_id, family, phases in WORKLOADS:
        for variant in range(10):
            specs.append({
                "task_id": f"{workload_id}__{variant:02d}",
                "template_id": workload_id,
                "family": family,
                "variant": variant,
                "phases": [{"type": t, "target_required": bool(req)} for t,req in phases],
                "resource_namespace": f"{family}/{workload_id}/{variant:02d}",
                "target_ids": ["A","B"],
            })
    assert len(specs) == n
    return specs

task_specs = make_task_specs()
by_family = defaultdict(list)
for t in task_specs:
    by_family[t["family"]].append(t["task_id"])

task_ids = []
for family in ["software","deployment","data","transactional"]:
    task_ids.extend(sorted(by_family[family]))

if RUN_MODE == "full":
    family_counts = {"software":8,"deployment":8,"data":7,"transactional":7}
    e0a_task_ids=[]
    for family,k in family_counts.items():
        e0a_task_ids.extend(sorted(by_family[family])[:k])
    selected_specs=task_specs
else:
    selected_specs=task_specs[:TASK_COUNT]
    e0a_task_ids=[t["task_id"] for t in selected_specs]

with (TASKS_DIR/"tasks.jsonl").open("w",encoding="utf-8") as f:
    for t in selected_specs:
        f.write(json.dumps(t)+"\n")
(MANIFESTS_DIR/"e0a_task_ids.json").write_text(json.dumps(e0a_task_ids,indent=2),encoding="utf-8")


In [ ]:

IMMUTABLE_EFFECTS = {"PUBLISH","APPEND"}
EFFECT_ACTIONS = {"VALIDATE","PUBLISH","PROMOTE","ROLLOUT","APPROVE","UPDATE","COMMIT","WRITE","APPEND"}
def make_plan(task):
    return [{"type": p["type"], "object": "A"} for p in task["phases"]]
def validate_plan(task, plan, start_phase=0):
    if len(plan) != len(task["phases"])-start_phase:
        return False
    for i, action in enumerate(plan, start=start_phase):
        phase = task["phases"][i]
        if action["type"] != phase["type"] or (phase["target_required"] and action["object"] not in {"A","B"}):
            return False
    return True
def make_effect(scenario_id, step, action):
    return {
        "effect_id": f"{scenario_id}:e{step:02d}",
        "step": int(step),
        "type": action["type"],
        "object": action["object"],
        "resource": f"resource:{action['object']}",
        "historical_fact": True,
        "postcondition": f"{action['type']}({action['object']})",
        "anchor": f"anchor:{scenario_id}:{step:02d}",
        "utility": 1.0 + 0.01*step,
    }
def history_for(scenario_id, plan, fault_step):
    return [make_effect(scenario_id,i,a) for i,a in enumerate(plan[:fault_step]) if a["type"] in EFFECT_ACTIONS]
def choose_fault(task, plan, seed):
    candidates=[]
    for fault_step in range(len(plan)-MIN_HORIZON):
        effects=history_for("fault",plan,fault_step)
        suffix=plan[fault_step+1:]
        types={e["type"] for e in effects}
        if len(effects)>=2 and len(suffix)>=MIN_HORIZON and types & IMMUTABLE_EFFECTS and types-IMMUTABLE_EFFECTS:
            candidates.append(fault_step)
    if not candidates: raise RuntimeError(f"No valid fault point for {task['task_id']}")
    return int(rng_for("fault",task["task_id"],seed).choice(candidates))


In [ ]:

def prompt_messages(task, fault_step, effects, suffix, a_branch, b_branch):
    next_type = suffix[0]["type"]
    return [
        {"role":"system","content":"Choose between two semantically different, already-valid recovery alternatives. Return exactly one option label."},
        {"role":"user","content":(
            f"Workload: {task['template_id']}\n"
            f"Family: {task['family']}\n"
            f"Fault boundary: after step {fault_step}\n\n"
            "Historical external effects:\n" +
            "\n".join(f"- {e['type']}({e['object']})" for e in effects) +
            "\n\nOption A semantics:\n"
            f"- recovered branch target: {a_branch}\n- next valid action: {next_type}({a_branch})\n\n"
            "Option B semantics:\n"
            f"- recovered branch target: {b_branch}\n- next valid action: {next_type}({b_branch})\n\n"
            "Choose exactly one option: A or B."
        )}
    ]

def _tokenize_prompt(messages):
    encoded = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    if isinstance(encoded, dict): ids=encoded["input_ids"]
    elif hasattr(encoded,"input_ids"): ids=encoded.input_ids
    else: ids=encoded
    if ids.ndim==1: ids=ids.unsqueeze(0)
    return ids

def score_completion(messages, completion):
    prompt_ids = _tokenize_prompt(messages).to(next(model.parameters()).device)
    comp = tokenizer(completion, add_special_tokens=False, return_tensors="pt")["input_ids"].to(prompt_ids.device)
    if comp.ndim==1: comp=comp.unsqueeze(0)
    full=torch.cat([prompt_ids[0].reshape(-1), comp[0].reshape(-1)], dim=0).unsqueeze(0)
    mask=torch.ones_like(full)
    with torch.no_grad():
        logits=model(input_ids=full, attention_mask=mask, use_cache=False).logits[0]
    p=prompt_ids.shape[1]
    n=comp.shape[1]
    positions=torch.arange(p-1,p-1+n,device=logits.device)
    logp=torch.log_softmax(logits[positions],dim=-1)
    chosen=logp[torch.arange(n,device=logits.device),comp[0]]
    return float(chosen.mean().detach().cpu())

def decide_branch(task, fault_step, effects, suffix, seed):
    if TEST_MODE:
        raw = math.sin(stable_hash((task["task_id"], fault_step, seed)) % 1000)
        m1 = float(raw/10.0)
        m2 = float(raw/12.0)
        cb = (m1+m2)/2
    else:
        m1 = score_completion(prompt_messages(task,fault_step,effects,suffix,"A","B"),"A") - score_completion(prompt_messages(task,fault_step,effects,suffix,"A","B"),"B")
        m2 = score_completion(prompt_messages(task,fault_step,effects,suffix,"B","A"),"B") - score_completion(prompt_messages(task,fault_step,effects,suffix,"B","A"),"A")
        cb = (m1+m2)/2
    label_bias=(m1-m2)/2
    p_continue=float(np.clip(1/(1+math.exp(-cb/DECISION_TEMPERATURE)),1e-4,1-1e-4))
    branch="A" if rng_for("branch",task["task_id"],seed).random()<p_continue else "B"
    return {"recovery_branch":branch,"mapping1_margin":m1,"mapping2_margin":m2,"counterbalanced_margin":cb,
            "prob_continue":p_continue,"prob_switch":1-p_continue,"label_bias":label_bias,
            "source":"test_backend" if TEST_MODE else current_model["label"],"fallback":False}

def build_case(task, model_spec, seed):
    scenario_id=f"{task['task_id']}__{model_spec['label']}__{seed}"
    plan=make_plan(task)
    if not validate_plan(task, plan): raise RuntimeError(f"Invalid original plan: {scenario_id}")
    fault=choose_fault(task,plan,seed)
    suffix=plan[fault+1:]
    effects=history_for(scenario_id,plan,fault)
    decision=decide_branch(task,fault,effects,suffix,seed)
    branch=decision["recovery_branch"]
    recovered=[{"type":a["type"],"object":branch} for a in task["phases"][fault+1:]]
    if not validate_plan(task,recovered,start_phase=fault+1): raise RuntimeError(f"Invalid recovery plan: {scenario_id}")
    history=[]
    for e in effects:
        compatible = True if e["type"] in IMMUTABLE_EFFECTS else e["object"]==branch
        history.append({**e,"compatible":bool(compatible),"fact_persists":True,"state":"ADOPT" if compatible else "BLOCK"})
    valid=sum(r["state"]=="ADOPT" for r in history)
    invalid=sum(r["state"]=="BLOCK" for r in history)
    action_div = any(a!=b for a,b in zip(suffix,recovered))
    effect_div = any((a["type"] in EFFECT_ACTIONS or b["type"] in EFFECT_ACTIONS) and a!=b for a,b in zip(suffix,recovered))
    if effect_div and not action_div: raise RuntimeError(f"Effect/action invariant violated: {scenario_id}")
    case={
        "scenario_id":scenario_id,"task_id":task["task_id"],"template_id":task["template_id"],"family":task["family"],
        "model":model_spec["label"],"model_id":model_spec["model_id"],"seed":seed,"fault_step":fault,"horizon_K":len(suffix),
        "original_semantically_valid":True,"recovered_semantically_valid":True,
        "original_plan":plan,"original_postfault_actions":suffix,"recovered_actions":recovered,
        "history":history,"historical_effects":len(history),"valid_historical_effects":valid,"invalid_historical_effects":invalid,
        "mixed_validity":bool(valid>0 and invalid>0),"target_regime":bool(valid>0 and invalid>0),
        "action_divergence":bool(action_div),"effect_relevant_divergence":bool(effect_div),
        **decision,
        "generator_version":"cera-b0-vfinal","decision_temperature":DECISION_TEMPERATURE,
    }
    return case


In [ ]:

def load_model(spec):
    if TEST_MODE:
        return None
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for real model generation.")
    tokenizer_obj=AutoTokenizer.from_pretrained(spec["model_id"], trust_remote_code=True, token=os.environ.get("HF_TOKEN"))
    qcfg=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
    model_obj=AutoModelForCausalLM.from_pretrained(spec["model_id"],quantization_config=qcfg,device_map="auto",torch_dtype=torch.float16,trust_remote_code=True,token=os.environ.get("HF_TOKEN"))
    model_obj.eval()
    return tokenizer_obj, model_obj

def existing_cases(manifest_path):
    if not manifest_path.exists(): return {}
    out={}
    with manifest_path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r=json.loads(line); out[r["scenario_id"]]=r
    return out

tasks=[t for t in (json.loads(x) for x in (TASKS_DIR/"tasks.jsonl").read_text().splitlines())]
tasks=tasks[:TASK_COUNT]
case_manifest=MANIFESTS_DIR/("b0_case_manifest.jsonl")
done_cases=existing_cases(case_manifest)
total=len(tasks)*len(ACTIVE_MODELS)*len(ACTIVE_SEEDS)
completed=0
errors=[]
manifest_fh=case_manifest.open("a",encoding="utf-8")

for spec in ACTIVE_MODELS:
    current_model=spec
    tokenizer=model=None
    if not TEST_MODE:
        tokenizer,model=load_model(spec)
    for task in tasks:
        for seed in ACTIVE_SEEDS:
            sid=f"{task['task_id']}__{spec['label']}__{seed}"
            if sid in done_cases and Path(done_cases[sid]["trace_path"]).exists() and Path(done_cases[sid]["oracle_path"]).exists():
                completed+=1
                print(f"[{completed}/{total}] {sid} cached", flush=True)
                continue
            try:
                case=build_case(task,spec,seed)
                trace_dir=TRACES_DIR/task["task_id"]/spec["label"]
                oracle_dir=ORACLES_DIR/task["task_id"]/spec["label"]
                trace_dir.mkdir(parents=True,exist_ok=True); oracle_dir.mkdir(parents=True,exist_ok=True)
                trace_path=trace_dir/f"seed_{seed}.json"
                oracle_path=oracle_dir/f"seed_{seed}.json"
                trace_payload={k:v for k,v in case.items() if k not in {"valid_historical_effects","invalid_historical_effects"}}
                trace_path.write_text(json.dumps(trace_payload,indent=2),encoding="utf-8")
                oracle_path.write_text(json.dumps({"scenario_id":case["scenario_id"],"historical_effects":case["history"],"compatibility":{e["effect_id"]:e["state"] for e in case["history"]}},indent=2),encoding="utf-8")
                manifest_fh.write(json.dumps({"scenario_id":sid,"task_id":task["task_id"],"model":spec["label"],"seed":seed,"trace_path":str(trace_path),"oracle_path":str(oracle_path)})+"\n"); manifest_fh.flush()
                completed+=1
                print(f"[{completed}/{total}] {sid} margin={case['counterbalanced_margin']:.3f} branch={case['recovery_branch']}", flush=True)
            except Exception as exc:
                errors.append({"scenario_id":sid,"error":repr(exc)})
                print(f"[{completed}/{total}] ERROR {sid}: {exc}", flush=True)
    del model,tokenizer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
manifest_fh.close()
if errors:
    (MANIFESTS_DIR/"b0_errors.json").write_text(json.dumps(errors,indent=2),encoding="utf-8")
    if RUN_MODE=="full":
        raise RuntimeError(f"{len(errors)} benchmark cases failed. See {MANIFESTS_DIR/'b0_errors.json'} and resume after fixing the issue.")


In [ ]:

manifest_rows=[json.loads(x) for x in case_manifest.read_text().splitlines() if x.strip()]
if len(manifest_rows)!=total:
    raise RuntimeError(f"Expected {total} frozen cases, found {len(manifest_rows)}.")
df=pd.DataFrame(manifest_rows)
if df.duplicated("scenario_id").any(): raise RuntimeError("Duplicate scenario IDs in manifest.")
for _,r in df.iterrows():
    if not Path(r.trace_path).exists() or not Path(r.oracle_path).exists(): raise RuntimeError(f"Missing artifact for {r.scenario_id}")
loaded=[]
for _,r in df.iterrows():
    trace=json.loads(Path(r.trace_path).read_text())
    oracle=json.loads(Path(r.oracle_path).read_text())
    loaded.append({**trace,"oracle_labels":oracle["compatibility"]})
ldf=pd.DataFrame(loaded)
for col in ["counterbalanced_margin","prob_continue","label_bias"]:
    if not np.isfinite(ldf[col].astype(float)).all(): raise RuntimeError(f"Non-finite {col}")
if not ((ldf["prob_continue"]>0)&(ldf["prob_continue"]<1)).all(): raise RuntimeError("Invalid probabilities.")
if not set(ldf["recovery_branch"]).issubset({"A","B"}): raise RuntimeError("Invalid branch.")
qwen=ldf[ldf.model=="qwen3-8b"]
if len(qwen):
    qwen_diag={
        "cases":len(qwen),
        "both_branches_observed":qwen.recovery_branch.nunique()==2,
        "margin_std":float(qwen.counterbalanced_margin.std(ddof=1)) if len(qwen)>1 else 0.0,
        "prob_std":float(qwen.prob_continue.std(ddof=1)) if len(qwen)>1 else 0.0,
        "mean_abs_margin":float(qwen.counterbalanced_margin.abs().mean()),
        "mean_abs_label_bias":float(qwen.label_bias.abs().mean()),
    }
else:qwen_diag={}
task_family=pd.Series({r["task_id"]:r["family"] for r in tasks})
summary={"mode":RUN_MODE,"cases":len(ldf),"models":ACTIVE_MODELS,"seeds":ACTIVE_SEEDS,"qwen_diagnostic":qwen_diag,
         "family_counts":ldf.family.value_counts().to_dict(),"branch_counts":ldf.recovery_branch.value_counts().to_dict()}
(MANIFESTS_DIR/"b0_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
ldf.to_csv(MANIFESTS_DIR/"b0_case_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(7,4))
ldf.groupby("model")["counterbalanced_margin"].mean().plot(kind="bar",ax=ax)
ax.set_ylabel("Mean counterbalanced semantic margin"); ax.set_title("CERA-Bench generation diagnostics"); fig.tight_layout()
fig.savefig(MANIFESTS_DIR/"b0_generation_diagnostic.png",dpi=180,bbox_inches="tight"); plt.show()
print(json.dumps(summary,indent=2))
